# 03 Knowledge-Enhanced Agent

This notebook adds domain knowledge to candidate selection and compares recommendations before and after knowledge enhancement.

In [ ]:
import importlib
import os
from pathlib import Path

import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

import knowledge_base
import llm_agent_core

importlib.reload(knowledge_base)
importlib.reload(llm_agent_core)

from knowledge_base import (
    amino_acid_properties,
    apply_mutation_count_constraint,
    build_knowledge_context,
    build_mutation_knowledge_graph,
    score_mutation_rules,
    weighted_knowledge_score,
)
from llm_agent_core import add_mutation_features, build_designer_pool, knowledge_enhanced_scientific_critic, load_two_vs_many, merge_final_recommendations, summarize_history

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
MAX_MUTATIONS_FOR_DESIGN = 4
TOP_K = 10
RAW_POOL_SIZE = 5000
RUN_OPENAI = bool(os.getenv("OPENAI_API_KEY"))
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

## Amino-Acid Property Knowledge

The rule base keeps physicochemical properties used by the knowledge-enhanced ranking.

In [ ]:
properties_df = pd.DataFrame.from_dict(amino_acid_properties(), orient="index")
properties_df.index.name = "amino_acid"
display(properties_df)

## Optional LLM Setup

By default, `RUN_OPENAI` becomes `True` when `.env` contains `OPENAI_API_KEY`. If the API is rate-limited, the notebook falls back to the local rule critic.

In [ ]:
client = None
if RUN_OPENAI:
    from openai import OpenAI
    client = OpenAI()

print("RUN_OPENAI:", RUN_OPENAI)
print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY") is not None)
print("LLM model:", LLM_MODEL)


## Build Unenhanced Candidate Pool

This reproduces the historical-prior ranking before adding mutation-count and property rules.

In [ ]:
observed_df, candidate_pool = load_two_vs_many("two_vs_many.csv")
top_history, mutation_summary = summarize_history(observed_df)
unenhanced_pool = build_designer_pool(candidate_pool, mutation_summary, pool_size=RAW_POOL_SIZE)

test_truth = pd.read_csv("test.csv").reset_index(drop=True)
test_truth["candidate_id"] = [f"C{i:05d}" for i in range(len(test_truth))]
test_truth = test_truth.rename(columns={"target": "true_fitness"})

predictions = pd.read_csv(ARTIFACTS / "test_esm2_mlp_predictions.csv").reset_index(drop=True)
predictions["candidate_id"] = [f"C{i:05d}" for i in range(len(predictions))]
predictions = predictions.rename(columns={"predicted_fitness": "esm_predicted_fitness"})

def attach_eval_columns(df):
    result = df.merge(test_truth[["candidate_id", "true_fitness"]], on="candidate_id", how="left")
    result = result.merge(predictions[["candidate_id", "esm_predicted_fitness"]], on="candidate_id", how="left")
    result["esm_prediction_error"] = result["esm_predicted_fitness"] - result["true_fitness"]
    return result

unenhanced_top = attach_eval_columns(unenhanced_pool.head(TOP_K).copy())
unenhanced_top.insert(0, "rank", range(1, len(unenhanced_top) + 1))

display(unenhanced_top[[
    "rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "historical_prior",
    "esm_predicted_fitness",
    "true_fitness",
    "esm_prediction_error",
]])

## Knowledge-Enhanced Ranking

Knowledge enhancement adds mutation-count constraints, physicochemical rule scores, and normalized weighted scoring.

In [ ]:
knowledge_pool = apply_mutation_count_constraint(unenhanced_pool, max_mutations=MAX_MUTATIONS_FOR_DESIGN)
rule_scores = knowledge_pool["mutations"].apply(score_mutation_rules).apply(pd.Series)
knowledge_pool = pd.concat([knowledge_pool.reset_index(drop=True), rule_scores.reset_index(drop=True)], axis=1)
knowledge_pool = weighted_knowledge_score(knowledge_pool, historical_weight=0.7, rule_weight=0.3)
knowledge_pool = knowledge_pool.sort_values("knowledge_enhanced_score", ascending=False).reset_index(drop=True)

knowledge_top = attach_eval_columns(knowledge_pool.head(TOP_K).copy())
knowledge_top.insert(0, "rank", range(1, len(knowledge_top) + 1))
knowledge_top.to_csv(ARTIFACTS / "knowledge_enhanced_candidates.csv", index=False)

print("Max mutations for knowledge-enhanced design:", MAX_MUTATIONS_FOR_DESIGN)
print("Knowledge-enhanced candidate pool:", len(knowledge_pool))
display(knowledge_top[[
    "rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "historical_prior",
    "rule_score",
    "historical_prior_norm",
    "rule_score_norm",
    "knowledge_enhanced_score",
    "esm_predicted_fitness",
    "true_fitness",
    "esm_prediction_error",
]])

## Before vs After Knowledge Enhancement

This table directly compares the recommended sequences before and after adding knowledge rules.

In [ ]:
before_after = pd.concat([
    unenhanced_top.assign(strategy="before_knowledge"),
    knowledge_top.assign(strategy="after_knowledge"),
], ignore_index=True)
before_after.to_csv(ARTIFACTS / "knowledge_before_after_comparison.csv", index=False)

summary = (
    before_after
    .groupby("strategy")
    .agg(
        mean_num_mutations=("num_mutations", "mean"),
        mean_predicted_fitness=("esm_predicted_fitness", "mean"),
        mean_true_fitness=("true_fitness", "mean"),
        best_true_fitness=("true_fitness", "max"),
        mean_prediction_error=("esm_prediction_error", "mean"),
    )
    .reset_index()
)
summary.to_csv(ARTIFACTS / "knowledge_before_after_summary.csv", index=False)

display(before_after[[
    "strategy",
    "rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "historical_prior",
    "esm_predicted_fitness",
    "true_fitness",
    "esm_prediction_error",
]])
display(summary)

## Simple Knowledge Graph Triples

The following triples make the knowledge layer explicit: amino-acid properties, mutation positions, mutation-fitness observations, and variant-mutation membership.

In [ ]:
kg_input = knowledge_top[["candidate_id", "mutations", "true_fitness"]].to_dict(orient="records")
triples = build_mutation_knowledge_graph(kg_input)
triples_df = pd.DataFrame(triples, columns=["subject", "predicate", "object"])
triples_df.to_csv(ARTIFACTS / "mutation_knowledge_graph_triples.csv", index=False)

display(triples_df.head(40))

## Knowledge-Enhanced LLM Critic

This step sends the knowledge-enhanced candidates, amino-acid rules, knowledge-graph triples, predicted fitness, and benchmark true fitness to the LLM for a second scientific review.

In [ ]:
def build_knowledge_critic_candidates(df):
    columns = [
        "rank",
        "candidate_id",
        "mutations",
        "num_mutations",
        "historical_prior",
        "rule_score",
        "knowledge_enhanced_score",
        "esm_predicted_fitness",
        "true_fitness",
        "esm_prediction_error",
    ]
    records = []
    for _, row in df[columns].iterrows():
        records.append({
            "rank": int(row["rank"]),
            "candidate_id": str(row["candidate_id"]),
            "mutations": list(row["mutations"]),
            "num_mutations": int(row["num_mutations"]),
            "historical_prior": float(row["historical_prior"]),
            "rule_score": float(row["rule_score"]),
            "knowledge_enhanced_score": float(row["knowledge_enhanced_score"]),
            "predicted_fitness": float(row["esm_predicted_fitness"]),
            "true_fitness": float(row["true_fitness"]),
            "prediction_error": float(row["esm_prediction_error"]),
        })
    return records

knowledge_context = "\n\n".join(
    build_knowledge_context(mutations)
    for mutations in knowledge_top["mutations"].head(TOP_K)
)
knowledge_context += "\n\nSelection rule: keep candidates with no more than " + str(MAX_MUTATIONS_FOR_DESIGN) + " mutations, then rank by normalized historical_prior and physicochemical rule_score."

critic_client = client if RUN_OPENAI else None
if critic_client is None:
    class _UnavailableResponses:
        def parse(self, **kwargs):
            raise RuntimeError("RUN_OPENAI is False; using local fallback critic.")
    class _UnavailableClient:
        responses = _UnavailableResponses()
    critic_client = _UnavailableClient()

knowledge_critic_result = knowledge_enhanced_scientific_critic(
    critic_client,
    LLM_MODEL,
    build_knowledge_critic_candidates(knowledge_top),
    knowledge_context,
    triples_df,
)

knowledge_llm_recommendations = merge_final_recommendations(
    knowledge_top.rename(columns={"esm_predicted_fitness": "predicted_fitness"}),
    knowledge_critic_result,
)
knowledge_llm_recommendations.to_csv(ARTIFACTS / "knowledge_enhanced_llm_recommendations.csv", index=False)

display_columns = [
    "rank",
    "candidate_id",
    "mutations",
    "num_mutations",
    "knowledge_enhanced_score",
    "predicted_fitness",
    "true_fitness",
    "esm_prediction_error",
    "priority",
    "recommendation_reason",
    "supporting_evidence",
    "fitness_interpretation",
    "uncertainty",
]

display_df = knowledge_llm_recommendations[display_columns].copy()
with pd.option_context(
    "display.max_colwidth", None,
    "display.max_columns", None,
    "display.width", 0,
    "display.max_rows", None,
):
    display(
        display_df.style.set_properties(**{
            "white-space": "pre-wrap",
            "text-align": "left",
            "vertical-align": "top",
            "max-width": "520px",
        })
    )

print("========== KNOWLEDGE-ENHANCED OVERALL RECOMMENDATION ==========")
print(knowledge_critic_result.overall_recommendation)
print("\n========== KNOWLEDGE-ENHANCED LIMITATIONS ==========")
print(knowledge_critic_result.overall_limitations)


## Example Knowledge Context

This is the type of structured knowledge that can be injected into a prompt or used for rule-based filtering.

In [ ]:
example_mutations = knowledge_top.iloc[0]["mutations"]
print(build_knowledge_context(example_mutations))